In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [3]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems' # Enter dataset path
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] # Make the list of all genres available (alphabetical order)
STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav'] # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [4]:
import os
import random

def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    # --- Initialize Counters ---
    count_corrupted = 0
    count_less_5_0491 = 0
    count_greater_5_0493 = 0
    MB_IN_BYTES = 1024 * 1024
    # ---------------------------

    # Iterate through Genres
    for genre in GENRES:
        genre_dir = os.path.join(root_dir, genre)
        
        # Check: if genre folder exists
        if not os.path.isdir(genre_dir):
            continue

        valid_songs = []
        
        for song_name in os.listdir(genre_dir):
            song_path = os.path.join(genre_dir, song_name)
            
            if not os.path.isdir(song_path):
                continue

            is_complete = True
            is_not_corrupt = True
            
            # Check every stem
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                
                # CHECK : Completeness
                if not os.path.exists(stem_path):
                    is_complete = False
                    continue # Continue instead of break to count sizes of other stems
                
                # Get file size
                size_bytes = os.path.getsize(stem_path)
                
                # --- Update Counters ---
                if size_bytes < 4096:  # Less than 4KB
                    count_corrupted += 1
                    is_not_corrupt = False
                    
                if size_bytes < (5.0491 * MB_IN_BYTES):
                    count_less_5_0491 += 1
                    
                if size_bytes > (5.0493 * MB_IN_BYTES):
                    count_greater_5_0493 += 1
                # -----------------------

            # Only add to valid songs if it has all stems and none are corrupt
            if is_complete and is_not_corrupt:
                valid_songs.append(song_name)

        # Stratified Shuffle Split
        rng.shuffle(valid_songs)
        
        split_idx = int(len(valid_songs) * val_split)
        val_songs_list = valid_songs[:split_idx]
        train_songs_list = valid_songs[split_idx:]

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            for song in song_list:
                for stem in STEMS:
                    key = stem.replace('.wav', '')
                    full_path = os.path.join(genre_dir, song, stem)
                    target_dict[genre][key].append(full_path)

        # Populate the actual datasets
        add_to_dict(train_dataset, train_songs_list)
        add_to_dict(val_dataset, val_songs_list)

    # --- Print Assignment Answers ---
    print("--- DATASET BUILD COMPLETE ---")
    print(f"Total Corrupted (< 4KB): {count_corrupted}")
    print(f"Total < 5.0491 MB: {count_less_5_0491}")
    print(f"Total > 5.0493 MB: {count_greater_5_0493}\n")
    
    q1_ans = count_corrupted + count_less_5_0491
    q2_ans = abs(count_greater_5_0493 - count_less_5_0491)
    
    print(f"Q1 Answer [(Corrupted) + (< 5.0491MB)]: {q1_ans}")
    print(f"Q2 Answer [Absolute diff between > 5.0493MB and < 5.0491MB]: {q2_ans}")
    print("------------------------------")

    return train_dataset, val_dataset

# Execution
tr, val = build_dataset(DATA_ROOT)

# Q3 Calculation (Requires tr and val to be populated)
train_reggae_drums = len(tr['reggae']['drums'])
val_country_vocals = len(val['country']['vocals'])
q3_ans = abs(train_reggae_drums - val_country_vocals)

print(f"Q3 Answer [Absolute diff between Train Reggae Drums ({train_reggae_drums}) and Val Country Vocals ({val_country_vocals})]: {q3_ans}")

--- DATASET BUILD COMPLETE ---
Total Corrupted (< 4KB): 0
Total < 5.0491 MB: 1256
Total > 5.0493 MB: 184

Q1 Answer [(Corrupted) + (< 5.0491MB)]: 1256
Q2 Answer [Absolute diff between > 5.0493MB and < 5.0491MB]: 1072
------------------------------
Q3 Answer [Absolute diff between Train Reggae Drums (83) and Val Country Vocals (17)]: 66


In [5]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []
    # ------------------- write your code here -------------------------------

    total_files = sum(len(paths) for genre in dataset_dict.values() for paths in genre.values())     # ---- COUNT TOTAL FILES ----
    pbar = tqdm(total=total_files, desc="Analyzing Silence")
    for genre, stems in dataset_dict.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                
                # Load Audio
                try:
                    y, _ = librosa.load(file_path, sr=sr)
                    total_duration = librosa.get_duration(y=y, sr=sr)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    pbar.update(1)
                    continue

                # Find Non-Silent Intervals
                # intervals are returned in samples [[start, end], [start, end]]
                intervals = librosa.effects.split(y, top_db=top_db)
                
                max_silence = 0.0
                silence_type = []

                # CASE A: Fully silent
                if len(intervals) == 0:
                    max_silence = total_duration
                    silence_type.append("Full")
                else:
                    # Convert intervals from samples to seconds
                    intervals_sec = intervals / sr

                    # CASE B: START silence
                    start_gap = intervals_sec[0][0]
                    if start_gap > 0:
                        max_silence = max(max_silence, start_gap)
                        if start_gap >= threshold_sec:
                            silence_type.append("Start")

                    # CASE C: END silence
                    end_gap = total_duration - intervals_sec[-1][1]
                    if end_gap > 0:
                        max_silence = max(max_silence, end_gap)
                        if end_gap >= threshold_sec:
                            silence_type.append("End")

                    # CASE D: MIDDLE silence
                    # Check gaps between consecutive intervals
                    for i in range(len(intervals_sec) - 1):
                        # Start of next interval - End of current interval
                        mid_gap = intervals_sec[i+1][0] - intervals_sec[i][1]
                        if mid_gap > 0:
                            max_silence = max(max_silence, mid_gap)
                            if mid_gap >= threshold_sec:
                                if "Middle" not in silence_type: # Avoid duplicate "Middle" tags
                                    silence_type.append("Middle")

                # Store result
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(silence_type),
                        "File_Path": file_path
                    })
                
                pbar.update(1)
    
    pbar.close()
    #-------------------------------------------------------------------------
    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# --- RESULTS ANALYSIS ---

# ------------------- write your code here -------------------------------
if not df_silence.empty:
    print(f"\nFound {len(df_silence)} files with silence >= {DURATION}s.\n")
    
    # Pivot Table: Count by Genre vs Stem
    pivot_table = df_silence.pivot_table(
        index='Genre', 
        columns='Stem', 
        values='File_Path', 
        aggfunc='count', 
        fill_value=0
    )
    
    print("--- Breakdown of Silent Files by Genre and Stem ---")
    print(pivot_table)
    
    # Optional: Check 'Full' silence specifically (often indicates bad data)
    full_silence = df_silence[df_silence['Silence_Location'].str.contains("Full")]
    if not full_silence.empty:
        print("\n--- WARNING: Fully Silent Files Detected ---")
        print(full_silence[['Genre', 'Stem', 'File_Path']].head())
else:
    print("No files found with significant silence.")
#-------------------------------------------------------------------------

Analyzing Silence: 100%|██████████| 3320/3320 [07:58<00:00,  6.94it/s]



Found 658 files with silence >= 5.0s.

--- Breakdown of Silent Files by Genre and Stem ---
Stem       bass  drums  other  vocals
Genre                                
blues        16     23      5      41
classical    69     57      3      70
country      14     14      2      14
disco         6      2      2      17
hiphop       21      3     21       5
jazz         21     19      1      73
metal         4      0      1      37
pop          10      6      2       4
reggae        4      5      7      13
rock          9      7      0      30


In [6]:
if not df_silence.empty:
    print(f"Total files with silence >= 5s: {len(df_silence)}\n")
    
    # -------------------------------------------------------------------------
    # 1. What's the average Silence Length in Vocals (in secs)?
    # -------------------------------------------------------------------------
    vocals_df = df_silence[df_silence['Stem'] == 'vocals']
    if not vocals_df.empty:
        avg_silence_vocals = vocals_df['Max_Silence_Sec'].mean()
        print(f"-> Average Silence Length in Vocals: {avg_silence_vocals:.2f} seconds")
    else:
        print("-> Average Silence Length in Vocals: 0 (No vocal silences found)")

    # -------------------------------------------------------------------------
    # 2. Total number of drums sound tracks in jazz where silence >= 5 secs 
    #    and Silence_Location is ONLY middle.
    # -------------------------------------------------------------------------
    # Note: df_silence already only contains tracks with silence >= 5 secs.
    jazz_drums_mid_only = df_silence[
        (df_silence['Genre'] == 'jazz') & 
        (df_silence['Stem'] == 'drums') & 
        (df_silence['Silence_Location'] == 'Middle') # Exact match for 'Middle' only
    ]
    print(f"-> Jazz Drums (Silence >= 5s & ONLY Middle): {len(jazz_drums_mid_only)}")

    # -------------------------------------------------------------------------
    # 3. Total number of drums sound tracks in jazz where silence >= 5 secs 
    #    and Max_Silence_Sec >= 10.
    # -------------------------------------------------------------------------
    jazz_drums_max_10 = df_silence[
        (df_silence['Genre'] == 'jazz') & 
        (df_silence['Stem'] == 'drums') & 
        (df_silence['Max_Silence_Sec'] >= 10.0)
    ]
    print(f"-> Jazz Drums (Silence >= 10s): {len(jazz_drums_max_10)}\n")


    # =========================================================================
    # GENERALIZED ANALYSIS (ALL STEMS & GENRES)
    # =========================================================================
    print("-" * 50)
    print("GENERALIZED ANALYSIS: AVERAGE SILENCE BY STEM")
    print("-" * 50)
    avg_by_stem = df_silence.groupby('Stem')['Max_Silence_Sec'].mean().round(2)
    print(avg_by_stem.to_string())
    print("\n")

    print("-" * 50)
    print("GENERALIZED ANALYSIS: ONLY 'MIDDLE' SILENCE COUNT BY GENRE & STEM")
    print("-" * 50)
    # Filter for only Middle, then group
    only_middle_df = df_silence[df_silence['Silence_Location'] == 'Middle']
    if not only_middle_df.empty:
        mid_counts = only_middle_df.groupby(['Genre', 'Stem']).size().unstack(fill_value=0)
        print(mid_counts)
    else:
        print("No tracks found with ONLY middle silence.")
    print("\n")

    print("-" * 50)
    print("GENERALIZED ANALYSIS: SILENCE >= 10s COUNT BY GENRE & STEM")
    print("-" * 50)
    # Filter for >= 10s, then group
    over_10s_df = df_silence[df_silence['Max_Silence_Sec'] >= 10.0]
    if not over_10s_df.empty:
        over_10_counts = over_10s_df.groupby(['Genre', 'Stem']).size().unstack(fill_value=0)
        print(over_10_counts)
    else:
        print("No tracks found with silence >= 10 seconds.")

else:
    print("DataFrame is empty. No files met the silence threshold.")

Total files with silence >= 5s: 658

-> Average Silence Length in Vocals: 12.79 seconds
-> Jazz Drums (Silence >= 5s & ONLY Middle): 11
-> Jazz Drums (Silence >= 10s): 5

--------------------------------------------------
GENERALIZED ANALYSIS: AVERAGE SILENCE BY STEM
--------------------------------------------------
Stem
bass      12.97
drums     12.64
other      8.96
vocals    12.79


--------------------------------------------------
GENERALIZED ANALYSIS: ONLY 'MIDDLE' SILENCE COUNT BY GENRE & STEM
--------------------------------------------------
Stem       bass  drums  other  vocals
Genre                                
blues         7      7      3      16
classical    13     13      3      11
country       3      4      2       6
disco         2      0      0       9
hiphop        9      0     14       1
jazz          8     11      0      17
metal         1      0      1      13
pop           4      4      1       3
reggae        0      2      5       7
rock          2      2  

In [7]:
stems_audio = []
try:
    for key in STEM_KEYS:
    # ------------------- write your code here -------------------------------
    # Load audio (Duration 5.0s for speed/consistency)
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        y, _ = librosa.load(file_path, sr=SR, duration=5.0)
            
        stems_audio.append(y)
    #-------------------------------------------------------------------------

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [8]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.array(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2))

#Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------
print(f"RMS Amplitude: {rms_val:.4f}")
print("Mix normalized and validated successfully!")

RMS Amplitude: 0.1100
Mix normalized and validated successfully!
